# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR<sup>2</sup> dataset using the `mlcroissant` library.

### Dataset Source
This dataset is defined by a Croissant schema available at the following URL:

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata  # 'metadata' is an object, not a dict

print(f"Dataset Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Let's inspect which record sets are available, along with their `@id` fields, and the contained fields/columns by their `@id` as well.

In [ ]:
# List all record sets with their IDs and fields
record_sets = list(metadata.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set Name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print("  Fields / columns (@id's):")
    for field in rs.fields:
        print(f"    - {field.name}: {field.id}")
    print()
# For this dataset, usually only one record set.
record_set_ids = [rs.id for rs in record_sets]

## 3. Data Extraction
Let's extract all records from the main record set into a pandas DataFrame using the record set `@id`. Each column and field is referenced by its `@id` property.

In [ ]:
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded DataFrame with {len(dataframes[record_set_id])} rows and {len(dataframes[record_set_id].columns)} columns.")
    print(f"Columns available: {list(dataframes[record_set_id].columns)}\n")

main_record_set_id = record_set_ids[0]
df = dataframes[main_record_set_id]
print("First few records:")
df.head()

## 4. Exploratory Data Analysis (EDA)
Let's perform some common data processing: filtering, normalization, and grouping on specific fields, always referencing fields/columns using their `@id`.

In [ ]:
# Identify a numeric field (column) by @id for filtering & EDA
# Example: Use 'coverage' as peptide coverage percentage, typically the ID might be 'coverage' (adjust if necessary)

import numpy as np

# Attempt to auto-detect a numeric field (@id)
numeric_candidates = []
for c in df.columns:
    if pd.api.types.is_numeric_dtype(df[c]):
        numeric_candidates.append(c)
if not numeric_candidates:
    # Try to convert some likely columns to numeric
    for c in df.columns:
        # Heuristic: Looks for numbers in first few rows
        try:
            df[c] = pd.to_numeric(df[c], errors='ignore')
            if pd.api.types.is_numeric_dtype(df[c]):
                numeric_candidates.append(c)
        except:
            pass

print(f"Numeric field candidates: {numeric_candidates}")
if numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    raise Exception('No numeric field found for EDA.')

# Filter records for which the value in `numeric_field` exceeds a threshold
threshold = df[numeric_field].mean() if np.isfinite(df[numeric_field].mean()) else 10.0
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"Filtered DataFrame ({numeric_field} > {threshold}): {len(filtered_df)} records")
display(filtered_df.head())

# Normalize the numeric field
col_norm = f"{numeric_field}_normalized"
filtered_df[col_norm] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized column '{col_norm}':")
display(filtered_df[[numeric_field, col_norm]].head())

# Group by a categorical/text field (e.g., 'accession' or similar primary annotation).
group_field = None
for c in df.columns:
    if c != numeric_field and df[c].dtype == 'object':
        group_field = c
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
    print(f"Grouped mean of '{numeric_field}' by '{group_field}':")
    display(grouped_df.head())
else:
    print('No grouping field found.')

## 5. Visualization
Let's visualize the distribution of the selected numeric field and the normalized data. We'll use matplotlib and seaborn for quick visualizations.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axs = plt.subplots(1, 2, figsize=(12, 5))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True, ax=axs[0])
axs[0].set_title(f'Distribution of {numeric_field}')
axs[0].set_xlabel(numeric_field)

sns.histplot(filtered_df[col_norm].dropna(), bins=30, kde=True, ax=axs[1], color='orange')
axs[1].set_title(f'Normalized {numeric_field} (filtered)')
axs[1].set_xlabel(col_norm)

plt.tight_layout()
plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load and analyze the FAIR<sup>2</sup> dataset package "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" using the `mlcroissant` library. By referencing all entities via their `@id`, we've ensured reproducibility and transparency in data processing steps. You can further explore other fields and analytical approaches using this framework.